# 01 · API card (easy): EPIC, Earth from space

**NASA Space Apps Cairo 2026 · Bootcamp · API & Data Integration**

**Goal:** pick a date, list that day's whole-Earth photos from the EPIC camera and show an *Earth card*.

**Free key** · **Trap:** a date with no photos returns `200 OK` and an **empty list**.

> ✍️ **Starter notebook.** Cells marked `TODO` are yours to finish. Green ✅ check cells tell you when a step works.

Open in Colab: *File → Save a copy in Drive* first, so your edits are kept.

## Setup

In [ ]:
# ▶ Run this cell first
import os, json, requests
import pandas as pd
from datetime import date, timedelta

API_KEY = "DEMO_KEY"   # 👉 replace with YOUR free key from https://api.nasa.gov (DEMO_KEY runs out fast in a full room)
# Never paste a paid key (for example an AI key) into a notebook you share.

def check(test, good, hint):
    """Tiny self-check: prints ✅ when the step works, ❌ plus a hint when it does not."""
    try:
        ok = bool(test())
    except Exception:
        ok = False
    print(("✅ " + good) if ok else ("❌ " + hint))

def show_url(url, params):
    """Print a request URL with your key hidden, so it is safe to show on a projector."""
    req = requests.Request("GET", url, params=params).prepare()
    key = str(params.get("api_key", ""))
    print(req.url.replace(key, "YOUR_KEY") if key else req.url)

In [ ]:
# 🛟 Safety net: ILLUSTRATIVE sample in the shape of an EPIC answer (not a real photo list).
SAMPLE_EPIC = [
    {"image": "epic_1b_SAMPLE_1", "date": "2025-10-01 00:31:45", "caption": "SAMPLE caption (illustrative)",
     "centroid_coordinates": {"lat": 10.5, "lon": 20.5}},
    {"image": "epic_1b_SAMPLE_2", "date": "2025-10-01 02:20:30", "caption": "SAMPLE caption (illustrative)",
     "centroid_coordinates": {"lat": 10.4, "lon": -5.2}},
]

## 1 · Which dates have photos?
EPIC does not have a photo for every date. Ask the API which dates exist, then pick the **latest** one.

Endpoint: `https://api.nasa.gov/EPIC/api/natural/available`

### Your turn

In [ ]:
AVAILABLE_URL = "https://api.nasa.gov/EPIC/api/natural/available"
r = requests.get(AVAILABLE_URL, params={"api_key": API_KEY}, timeout=15)
dates = ...      # TODO: the response as a Python list of date strings
DAY = ...        # TODO: the latest date in that list
print(DAY)

In [ ]:
check(lambda: isinstance(DAY, str) and len(DAY) == 10, "Got a date like 2025-10-01.", "DAY should be one YYYY-MM-DD string.")

## 2 · List the day's photos
Endpoint: `https://api.nasa.gov/EPIC/api/natural/date/<YYYY-MM-DD>`

### Your turn

In [ ]:
URL = f"https://api.nasa.gov/EPIC/api/natural/date/{DAY}"
resp = requests.get(..., params=..., timeout=15)    # TODO
photos = ...                                         # TODO: the JSON list
print(resp.status_code, "->", len(photos), "photos")

In [ ]:
check(lambda: resp.status_code == 200 and len(photos) > 0, "200 OK and photos found.",
      "Need status 200 and at least one photo. Is your key right?")

## 3 · Pick the fields you need
For each photo keep: `image` (a file name), `date` (time taken), `caption`, and the Earth's centre latitude/longitude (`centroid_coordinates`).

Look at one photo first: `print(json.dumps(photos[0], indent=2))`

### Your turn

In [ ]:
rows = []
for p in photos:
    rows.append({
        "image": ...,
        "time": ...,
        "caption": ...,
        "lat": ...,     # inside centroid_coordinates
        "lon": ...,
    })
df = pd.DataFrame(rows)
df.head()

In [ ]:
check(lambda: len(df) == len(photos) and {"image", "time", "caption", "lat", "lon"} <= set(df.columns),
      "Clean table of photos.", "One row per photo, with image, time, caption, lat and lon.")

## 4 · Show an Earth card
The picture itself lives at a different address:

`https://api.nasa.gov/EPIC/archive/natural/<YYYY>/<MM>/<DD>/png/<image>.png?api_key=<key>`

Build that address for the first photo and show it with a caption.

💡 Before you share this notebook, use *Edit → Clear all outputs*: the image address contains your key.

### Your turn

In [ ]:
from IPython.display import Image, display

y, m, d = DAY.split("-")

def image_url(name):
    return ...        # TODO: build the archive address from y, m, d and name

p = df.iloc[0]
display(Image(url=image_url(p["image"]), width=420))
print(p["caption"])
print("Taken:", p["time"], "| centre of the Earth disc: lat", p["lat"], "lon", p["lon"])

In [ ]:
check(lambda: "archive/natural" in image_url("x") and DAY.replace("-", "/") in image_url("x"),
      "Card built from the archive address.", "The address needs /archive/natural/YYYY/MM/DD/png/<image>.png")

## 5 · Save a backup

### Your turn

In [ ]:
os.makedirs("backup", exist_ok=True)
# TODO: save the raw photo list as backup/epic_photos.json
# TODO: save the table as backup/epic_table.csv (no index)
print(os.listdir("backup"))

In [ ]:
check(lambda: os.path.exists("backup/epic_photos.json") and os.path.exists("backup/epic_table.csv"),
      "Backup saved.", "Save both backup files.")

## 6 · Beat the trap: a date with no photos
Try `photos_for("2000-01-01")`: EPIC only started in 2015, so you get `200 OK` with an **empty list**: no error, but no data.

Write `photos_for(day)` so that it: calls the API; if the list is empty, prints a clear message and **falls back to the latest available date**; returns `(photos, day_used)`.

### Your turn

In [ ]:
def photos_for(day):
    r = requests.get(f"https://api.nasa.gov/EPIC/api/natural/date/{day}", params={"api_key": API_KEY}, timeout=15)
    r.raise_for_status()
    photos = r.json()
    # TODO: if photos is empty: print a clear message, look up the latest available date,
    #       call the API again for that date and use that list instead
    return photos, day

In [ ]:
test_photos, used = photos_for("2000-01-01")
check(lambda: len(test_photos) > 0 and used != "2000-01-01", "Empty list handled with a clear message and a fallback.",
      "photos_for('2000-01-01') should print a message and return photos from another date.")

## 7 · Plan for a failed call
If the API is down or your key is wrong, your card should still show something. Wrap the call in `try/except`, print a clear message, then use `backup/epic_photos.json` (or the sample, with the warning).

### Your turn

In [ ]:
def load_photos(day, key):
    try:
        r = requests.get(f"https://api.nasa.gov/EPIC/api/natural/date/{day}", params={"api_key": key}, timeout=15)
        # TODO: raise ValueError with a clear message for 403 and 429, r.raise_for_status() for the rest
        return r.json(), "live"
    except (requests.RequestException, ValueError) as e:
        print("⚠️  API problem:", e)
        # TODO: saved copy if backup/epic_photos.json exists, else the sample with a warning
        ...

In [ ]:
_, source = load_photos(DAY, "WRONG_KEY")
check(lambda: source in ("saved copy", "sample"), "A wrong key no longer breaks your card.",
      "With a wrong key your function should print a message and return saved or sample data.")

## ✅ Done means
- [ ] a `200 OK` call with your key
- [ ] a clean table
- [ ] saved JSON and CSV
- [ ] one Earth card on screen
- [ ] the empty-list trap and a failed call handled

**Stretch:** show *all* of the day's photos as a small gallery, ordered by time.